# **CryptArithmetic**

SEND + MORE = MONEY

ISA + ROA = TELO

In [1]:
var = ['I','S','A','R','O','T','E','L']
dom = [0,1,2,3,4,5,6,7,8,9]
cont = [(),(),(),(),()]

In [ ]:
import itertools
import time
import re

class CryptaSolver:
    def __init__(self, equation):
        # Nettoyage et analyse de l'équation (ex: "SEND + MORE = MONEY")
        self.equation = equation
        self.mots = re.findall(r'[A-Z]+', equation.upper())
        self.lettres_uniques = tuple(set("".join(self.mots)))
        self.premieres_lettres = set(m[0] for m in self.mots if len(m) > 1)

        if len(self.lettres_uniques) > 10:
            raise ValueError("Trop de lettres uniques (max 10).")

    def verifier(self, d):
        # Vérifie les zéros non significatifs
        if any(d[lettre] == 0 for lettre in self.premieres_lettres):
            return False

        # Transformation de l'équation en expression mathématique évaluable
        # On remplace chaque lettre par sa valeur dans le dictionnaire d
        expr = self.equation.upper()
        for lettre, valeur in d.items():
            expr = expr.replace(lettre, str(valeur))

        # On sépare le membre gauche et droit de l'égalité
        gauche, droite = expr.split('=')
        try:
            return eval(gauche) == eval(droite)
        except:
            return False

    # 1. GENERATE AND TEST (Force brute pure)
    def solve_gt(self):
        start = time.time()
        count = 0
        # On teste toutes les permutations possibles
        for p in itertools.permutations(range(10), len(self.lettres_uniques)):
            count += 1
            d = dict(zip(self.lettres_uniques, p))
            if self.verifier(d):
                return d, count, time.time() - start
        return None, count, time.time() - start

    # 2. BACKTRACKING
    def solve_bt(self, assignation=None, count=0):
        if assignation is None: assignation = {}

        if len(assignation) == len(self.lettres_uniques):
            return (assignation, count) if self.verifier(assignation) else (None, count)

        lettre = self.lettres_uniques[len(assignation)]
        for chiffre in range(10):
            # Contrainte all_different
            if chiffre not in assignation.values():
                assignation[lettre] = chiffre
                res, count = self.solve_bt(assignation, count + 1)
                if res: return res, count
                del assignation[lettre]
        return None, count

    # 3. FORWARD CHECKING
    def solve_fc(self, assignation=None, domaines=None, count=0):
        if assignation is None: assignation = {}
        if domaines is None: domaines = {l: list(range(10)) for l in self.lettres_uniques}

        if len(assignation) == len(self.lettres_uniques):
            return (assignation, count) if self.verifier(assignation) else (None, count)

        lettre = self.lettres_uniques[len(assignation)]

        for chiffre in domaines[lettre]:
            # Contrainte du premier chiffre != 0
            if chiffre == 0 and lettre in self.premieres_lettres:
                continue

            # Forward Checking : On prépare les domaines futurs
            nouveaux_domaines = {l: [c for c in d if c != chiffre] for l, d in domaines.items()}
            assignation[lettre] = chiffre

            # Si aucune variable future n'a son domaine vide, on continue
            if all(len(d) > 0 for l, d in nouveaux_domaines.items() if l not in assignation):
                res, count = self.solve_fc(assignation, nouveaux_domaines, count + 1)
                if res: return res, count

            del assignation[lettre]
        return None, count

In [4]:
equa = "SEND + MORE = MONEY"
solver = CryptaSolver(equa)

print(f"Résolution de : {equa}")
print("-" * 50)
print("Méthode BT...")
sol_bt, c_bt = solver.solve_bt()
print(f"Trouvé en {c_bt} essais : {sol_bt}")

print("\nMéthode FC...")
sol_fc, c_fc = solver.solve_fc()
print(f"Trouvé en {c_fc} essais : {sol_fc}")

Résolution de : SEND + MORE = MONEY
--------------------------------------------------
Méthode BT...
Trouvé en 1712898 essais : {'N': 6, 'E': 5, 'M': 1, 'Y': 2, 'O': 0, 'D': 7, 'S': 9, 'R': 8}

Méthode FC...
Trouvé en 1402158 essais : {'N': 6, 'E': 5, 'M': 1, 'Y': 2, 'O': 0, 'D': 7, 'S': 9, 'R': 8}


In [5]:
equa = "ISA + ROA = TELO"
solver = CryptaSolver(equa)

print(f"Résolution de : {equa}")
print("-" * 50)
print("Méthode BT...")
sol_bt, c_bt = solver.solve_bt()
print(f"Trouvé en {c_bt} essais : {sol_bt}")

print("\nMéthode FC...")
sol_fc, c_fc = solver.solve_fc()
print(f"Trouvé en {c_fc} essais : {sol_fc}")

Résolution de : ISA + ROA = TELO
--------------------------------------------------
Méthode BT...
Trouvé en 292808 essais : {'T': 1, 'A': 2, 'E': 0, 'L': 9, 'O': 4, 'I': 3, 'S': 5, 'R': 7}

Méthode FC...
Trouvé en 32158 essais : {'T': 1, 'A': 2, 'E': 0, 'L': 9, 'O': 4, 'I': 3, 'S': 5, 'R': 7}
